In [140]:
import pandas as pd
import geopandas as gpd
import os

dataset_folder = r"C:\Users\michael\Desktop\IDX Exchange\Files\datasets"
os.chdir(dataset_folder)

In [141]:
df_sold = pd.read_csv("CRMLSSold_4.csv", low_memory = False)
df_listings = pd.read_csv("CRMLSListing_4.csv", low_memory = False)

In [142]:
df_sold.head()

,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,CloseDate,ClosePrice,ListAgentFirstName,...,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,OriginatingSystemName,OriginatingSystemSubName,year_month,rate_30yr_fixed,geometry
0,ContraCosta,ContraCosta,"Tile,Wood",NaN,False,5000000.0,1095075487,2024-01-18,5000000.0,Erin,...,3.0,NaN,94595,NaN,30240.0,CRMLS,CRMLS_CCBE,2024-01,6.6425,POINT (-122.08132056 37.88530534)
1,NaN,Mlslistings,NaN,False,NaN,NaN,1079166779,2024-01-30,858000.0,David,...,2.0,Palm Springs Unified,92262,NaN,13504.0,CRMLS,CRMLS_MLSL,2024-01,6.6425,POINT (-116.54053587 33.84982832)
2,Glendale,Southland,NaN,False,False,1890500.0,1075037759,2024-01-29,1890500.0,Karen,...,2.0,Los Angeles Unified,91356,0.0,17873.0,CRMLS,CRMLS_CRM,2024-01,6.6425,POINT (-118.558749 34.164376)
3,NorthSanLuisObispo,NorthSanLuisObispo,NaN,True,False,2100000.0,1067652762,2024-01-02,2100000.0,Jennifer,...,3.0,San Luis Coastal Unified,93401,0.0,11219.0,CRMLS,CRMLS_CRM,2024-01,6.6425,POINT (-120.646786 35.277199)
4,Mlslistings,Mlslistings,"Tile,Wood",True,NaN,NaN,1061988701,2024-01-31,2340000.0,Danny,...,3.0,Other,95148,NaN,8712.0,CRMLS,CRMLS_MLSL,2024-01,6.6425,POINT (-121.78025459 37.32544564)


In [143]:
df_listings.head()

,OriginalListPrice,ListingKey,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,PropertyType,...,GarageSpaces,HighSchoolDistrict,PostalCode,BuyerOfficeName.1,AssociationFee,LotSizeSquareFeet,UnparsedAddress.1,year_month,rate_30yr_fixed,geometry
0,759000.0,1159972295,NaN,NaN,Richard,Sirois,33.738910,-116.198447,42680 Incantata Place,Residential,...,2.0,Desert Sands Unified,92203,NaN,289.0,5227.0,42680 Incantata Place,2024-01,6.6425,POINT (-116.198447 33.73891)
1,320000.0,1159871345,2026-05-11,320000.0,Nicholas,Miller,33.842793,-116.481803,28388 Desert Princess Drive,Residential,...,1.0,NaN,92234,Coldwell Banker Realty,855.0,1307.0,28388 Desert Princess Drive,2024-01,6.6425,POINT (-116.481803 33.842793)
2,115900.0,1153320466,2026-04-10,160000.0,Heidi,Dunk-Vincent,39.125903,-122.861472,6900 Glenn,Residential,...,1.0,Upper Lake Union,95464,Noble Realty,0.0,3920.0,6900 Glenn,2024-01,6.6425,POINT (-122.861472 39.125903)
3,2500000.0,1146529264,2026-04-17,2000000.0,Clifford,Stevens,34.150964,-118.390267,4415 Morella Ave,Residential,...,2.0,Los Angeles Unified,91607,Equity Union,0.0,9906.0,4415 Morella Ave,2024-01,6.6425,POINT (-118.390267 34.150964)
4,3200000.0,1118382573,2025-05-21,3200000.0,Trudy,McGrath,33.028788,-117.264434,1200 Cardiff Drive,Residential,...,0.0,San Dieguito Union,92024,NonMember/Member-Other Board,0.0,54885.6,1200 Cardiff Drive,2024-01,6.6425,POINT (-117.264434 33.028788)


In [144]:
# When datasets are saved as CSVs, fields formatted as "datetime64[ns]" become "O". The following code converts them back to datetime format:
date_fields = ["CloseDate", "PurchaseContractDate", "ListingContractDate", "ContractStatusChangeDate"]

for field in date_fields:
    sold_old_type = df_sold[field].dtype
    df_sold[field] = pd.to_datetime(df_sold[field])
    print(f"{field} (Sold): {sold_old_type} -> {df_sold[field].dtype}")
    
    listings_old_type = df_listings[field].dtype    
    df_listings[field] = pd.to_datetime(df_listings[field])
    print(f"{field} (Listings): {listings_old_type} -> {df_listings[field].dtype}\n")

CloseDate (Sold): object -> datetime64[ns]
CloseDate (Listings): object -> datetime64[ns]

PurchaseContractDate (Sold): object -> datetime64[ns]
PurchaseContractDate (Listings): object -> datetime64[ns]

ListingContractDate (Sold): object -> datetime64[ns]
ListingContractDate (Listings): object -> datetime64[ns]

ContractStatusChangeDate (Sold): object -> datetime64[ns]
ContractStatusChangeDate (Listings): object -> datetime64[ns]



In [145]:
# Convert each property’s Latitude and Longitude into a geographic point
gdf_sold = gpd.GeoDataFrame(df_sold, 
                           geometry = gpd.points_from_xy(df_sold["Longitude"], df_sold["Latitude"]), 
                           crs = "EPSG:4326")

gdf_listings = gpd.GeoDataFrame(df_listings, 
                               geometry = gpd.points_from_xy(df_listings["Longitude"], df_listings["Latitude"]), 
                               crs = "EPSG:4326")

In [146]:
# GeoJSON of California's school districts was retrieved from "https://data.ca.gov/dataset/california-school-district-areas-2025-26", place this file in the "datasets" folder
gdf_school_districts = gpd.read_file("DistrictAreas2526_-284845464123469011.geojson")
gdf_school_districts = gdf_school_districts.to_crs("EPSG:4326")
gdf_school_districts = gdf_school_districts[gdf_school_districts.DistrictType == "Unified"]

In [147]:
# Perform a spatial join (gpd.sjoin) to determine which Unified School District polygon contains each property
df_sold = gpd.sjoin(gdf_sold, gdf_school_districts[["geometry", "DistrictName"]], how = "left")
df_sold = df_sold.drop(columns = ["index_right"])

df_listings = gpd.sjoin(gdf_listings, gdf_school_districts[["geometry", "DistrictName"]], how = "left")
df_listings = df_listings.drop(columns = ["index_right"])

In [148]:
print("Missing Rows (DistrictName)")
print(f"- Sold: {df_sold.DistrictName.isna().sum()} ({df_sold.DistrictName.isna().sum() / len(df_sold) * 100:.3f}%)")
print(f"- Listings: {df_listings.DistrictName.isna().sum()} ({df_listings.DistrictName.isna().sum() / len(df_listings) * 100:.3f}%)")

Missing Rows (DistrictName)
- Sold: 93960 (23.400%)
- Listings: 106115 (23.357%)


In [149]:
df_sold[["CountyOrParish", "DistrictName"]].head()

,CountyOrParish,DistrictName
0,Contra Costa,NaN
1,Riverside,Palm Springs Unified
2,Los Angeles,Los Angeles Unified
3,San Luis Obispo,San Luis Coastal Unified
4,Santa Clara,NaN


In [150]:
df_listings[["CountyOrParish", "DistrictName"]].head()

,CountyOrParish,DistrictName
0,Riverside,Desert Sands Unified
1,Riverside,Palm Springs Unified
2,Lake,Upper Lake Unified
3,Los Angeles,Los Angeles Unified
4,San Diego,NaN


In [151]:
# "price_ratio" - measures negotiation strength
def metric_price_ratio(df):
    df["price_ratio"] = df.ClosePrice / df.OriginalListPrice


# "PPSF" - normalizes price across sizes
def metric_PPSF(df):
    df["PPSF"] = df.ClosePrice / df.LivingArea


# "days_on_market" - time-to-sell indicator
def metric_days_on_market(df):
    df["days_on_market"] = df.DaysOnMarket


# "YrMo" - enables time-series analysis
def metric_YrMo(df):
    df["YrMo"] = pd.to_datetime(df.CloseDate).dt.to_period("M")


# "close_to_original_list_ratio" - captures full price reduction history
def metric_close_to_original_list_ratio(df):
    df["close_to_original_list_ratio"] = df.ClosePrice / df.OriginalListPrice


# "listing_to_contract_days" - measures time from listing to accepted offer
def metric_listing_to_contract_days(df):
    df["listing_to_contract_days"] = df.PurchaseContractDate - df.ListingContractDate
    df["listing_to_contract_days"] = df["listing_to_contract_days"].dt.days


# "contract_to_close_days" - escrow and closing period duration
def metric_contract_to_close_days(df):
    df["contract_to_close_days"] = df.CloseDate - df.PurchaseContractDate
    df["contract_to_close_days"] = df["contract_to_close_days"].dt.days

In [152]:
def all_metrics(df):
    metric_price_ratio(df)
    metric_PPSF(df)
    metric_days_on_market(df)
    metric_YrMo(df)
    metric_close_to_original_list_ratio(df)
    metric_listing_to_contract_days(df)
    metric_contract_to_close_days(df)

In [153]:
all_metrics(df_sold)
all_metrics(df_listings)

In [154]:
metrics_compare = ["ClosePrice", "OriginalListPrice", "price_ratio", "close_to_original_list_ratio", "LivingArea", "PPSF", "DaysOnMarket", "days_on_market", "CloseDate", "YrMo", "ListingContractDate", "PurchaseContractDate", "CloseDate", "listing_to_contract_days", "contract_to_close_days"]
metrics_only = ["price_ratio", "close_to_original_list_ratio", "PPSF", "days_on_market", "YrMo", "listing_to_contract_days", "contract_to_close_days"]

In [155]:
df_sold[metrics_compare].head()

,ClosePrice,OriginalListPrice,price_ratio,close_to_original_list_ratio,LivingArea,PPSF,DaysOnMarket,days_on_market,CloseDate,YrMo,ListingContractDate,PurchaseContractDate,CloseDate,listing_to_contract_days,contract_to_close_days
0,5000000.0,5000000.0,1.0,1.0,4354.0,1148.369316,0,0,2024-01-18,2024-01,2023-11-16,2023-11-16,2024-01-18,0.0,63.0
1,858000.0,NaN,NaN,NaN,1995.0,430.075188,0,0,2024-01-30,2024-01,2024-01-30,2024-01-30,2024-01-30,0.0,0.0
2,1890500.0,1890500.0,1.0,1.0,3194.0,591.891046,0,0,2024-01-29,2024-01,2024-01-29,2024-01-29,2024-01-29,0.0,0.0
3,2100000.0,2100000.0,1.0,1.0,3736.0,562.098501,0,0,2024-01-02,2024-01,2023-11-15,2023-11-15,2024-01-02,0.0,48.0
4,2340000.0,NaN,NaN,NaN,2442.0,958.230958,0,0,2024-01-31,2024-01,2024-01-31,2024-01-31,2024-01-31,0.0,0.0


In [156]:
df_listings[metrics_compare].head()

,ClosePrice,OriginalListPrice,price_ratio,close_to_original_list_ratio,LivingArea,PPSF,DaysOnMarket,days_on_market,CloseDate,YrMo,ListingContractDate,PurchaseContractDate,CloseDate,listing_to_contract_days,contract_to_close_days
0,NaN,759000.0,NaN,NaN,2338.0,NaN,36,36,NaT,NaT,2024-01-06,NaT,NaT,NaN,NaN
1,320000.0,320000.0,1.0000,1.0000,1212.0,264.026403,0,0,2026-05-11,2026-05,2024-01-11,2026-05-11,2026-05-11,851.0,0.0
2,160000.0,115900.0,1.3805,1.3805,1008.0,158.730159,10,10,2026-04-10,2026-04,2024-01-02,2026-03-16,2026-04-10,804.0,25.0
3,2000000.0,2500000.0,0.8000,0.8000,2573.0,777.302759,98,98,2026-04-17,2026-04,2024-01-24,2026-03-19,2026-04-17,785.0,29.0
4,3200000.0,3200000.0,1.0000,1.0000,1381.0,2317.161477,0,0,2025-05-21,2025-05,2024-01-28,2024-03-26,2025-05-21,58.0,421.0


In [157]:
df_sold[metrics_only].head()

,price_ratio,close_to_original_list_ratio,PPSF,days_on_market,YrMo,listing_to_contract_days,contract_to_close_days
0,1.0,1.0,1148.369316,0,2024-01,0.0,63.0
1,NaN,NaN,430.075188,0,2024-01,0.0,0.0
2,1.0,1.0,591.891046,0,2024-01,0.0,0.0
3,1.0,1.0,562.098501,0,2024-01,0.0,48.0
4,NaN,NaN,958.230958,0,2024-01,0.0,0.0


In [158]:
df_listings[metrics_only].head()

,price_ratio,close_to_original_list_ratio,PPSF,days_on_market,YrMo,listing_to_contract_days,contract_to_close_days
0,NaN,NaN,NaN,36,NaT,NaN,NaN
1,1.0000,1.0000,264.026403,0,2026-05,851.0,0.0
2,1.3805,1.3805,158.730159,10,2026-04,804.0,25.0
3,0.8000,0.8000,777.302759,98,2026-04,785.0,29.0
4,1.0000,1.0000,2317.161477,0,2025-05,58.0,421.0


In [159]:
# Segment Analysis
## PropertyType and PropertySubType
## CountyOrParish and MLSAreaMajor
## ListOfficeName and BuyerOfficeName



In [160]:
df_sold.head()

,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,CloseDate,ClosePrice,ListAgentFirstName,...,rate_30yr_fixed,geometry,DistrictName,price_ratio,PPSF,days_on_market,YrMo,close_to_original_list_ratio,listing_to_contract_days,contract_to_close_days
0,ContraCosta,ContraCosta,"Tile,Wood",NaN,False,5000000.0,1095075487,2024-01-18,5000000.0,Erin,...,6.6425,POINT (-122.08132 37.88531),NaN,1.0,1148.369316,0,2024-01,1.0,0.0,63.0
1,NaN,Mlslistings,NaN,False,NaN,NaN,1079166779,2024-01-30,858000.0,David,...,6.6425,POINT (-116.54054 33.84983),Palm Springs Unified,NaN,430.075188,0,2024-01,NaN,0.0,0.0
2,Glendale,Southland,NaN,False,False,1890500.0,1075037759,2024-01-29,1890500.0,Karen,...,6.6425,POINT (-118.55875 34.16438),Los Angeles Unified,1.0,591.891046,0,2024-01,1.0,0.0,0.0
3,NorthSanLuisObispo,NorthSanLuisObispo,NaN,True,False,2100000.0,1067652762,2024-01-02,2100000.0,Jennifer,...,6.6425,POINT (-120.64679 35.2772),San Luis Coastal Unified,1.0,562.098501,0,2024-01,1.0,0.0,48.0
4,Mlslistings,Mlslistings,"Tile,Wood",True,NaN,NaN,1061988701,2024-01-31,2340000.0,Danny,...,6.6425,POINT (-121.78025 37.32545),NaN,NaN,958.230958,0,2024-01,NaN,0.0,0.0


In [161]:
df_listings.head()

,OriginalListPrice,ListingKey,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,PropertyType,...,rate_30yr_fixed,geometry,DistrictName,price_ratio,PPSF,days_on_market,YrMo,close_to_original_list_ratio,listing_to_contract_days,contract_to_close_days
0,759000.0,1159972295,NaT,NaN,Richard,Sirois,33.738910,-116.198447,42680 Incantata Place,Residential,...,6.6425,POINT (-116.19845 33.73891),Desert Sands Unified,NaN,NaN,36,NaT,NaN,NaN,NaN
1,320000.0,1159871345,2026-05-11,320000.0,Nicholas,Miller,33.842793,-116.481803,28388 Desert Princess Drive,Residential,...,6.6425,POINT (-116.4818 33.84279),Palm Springs Unified,1.0000,264.026403,0,2026-05,1.0000,851.0,0.0
2,115900.0,1153320466,2026-04-10,160000.0,Heidi,Dunk-Vincent,39.125903,-122.861472,6900 Glenn,Residential,...,6.6425,POINT (-122.86147 39.1259),Upper Lake Unified,1.3805,158.730159,10,2026-04,1.3805,804.0,25.0
3,2500000.0,1146529264,2026-04-17,2000000.0,Clifford,Stevens,34.150964,-118.390267,4415 Morella Ave,Residential,...,6.6425,POINT (-118.39027 34.15096),Los Angeles Unified,0.8000,777.302759,98,2026-04,0.8000,785.0,29.0
4,3200000.0,1118382573,2025-05-21,3200000.0,Trudy,McGrath,33.028788,-117.264434,1200 Cardiff Drive,Residential,...,6.6425,POINT (-117.26443 33.02879),NaN,1.0000,2317.161477,0,2025-05,1.0000,58.0,421.0


In [163]:
# Save datasets as CSVs
## df_sold.to_csv("CRMLSSold_5.csv", index = False)
## df_listings.to_csv("CRMLSListing_5.csv", index = False)